In [2]:
!nvidia-smi

Thu Oct  1 22:52:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.82                 Driver Version: 592.82         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2050      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8              1W /   50W |     119MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [4]:
!pip install --upgrade accelerate
!pip uninstall -y transforers accelerate
!pip install transformers accelerate

Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0


  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)


In [5]:
!pip install evaluate

In [6]:
from transformers import pipeline, set_seed
from datasets import load_dataset , load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
from datasets import load_dataset
import evaluate

from transformers import AutoModelForSeq2SeqLM , AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

d:\Text-Summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anaya\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [8]:
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [9]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

d:\Text-Summarizer\textS\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anaya\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\Text-Summarizer\textS\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cac

In [10]:
# Load from Hugging Face Hub
dataset_samsum = load_dataset("knkarthick/samsum")

# Save the dataset to disk
dataset_samsum.save_to_disk("samsum_dataset")


d:\Text-Summarizer\textS\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anaya\.cache\huggingface\hub\datasets--knkarthick--samsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Saving the dataset (1/1 shards): 100%|██████████| 819/819 [00:00<00:00, 130722.85 examples/s]


In [11]:
dataset_samsum = load_from_disk("samsum_dataset")
print(dataset_samsum)


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [12]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths:{split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary")

print(dataset_samsum["test"][1]["summary"])

Split lengths:[14731, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary
Eric and Rob are going to watch a stand-up on youtube.


In [13]:
def convert_examples_to_features(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'] , max_length = 1024 ,truncation=True)
  target_encodings = tokenizer(
      text_target=example_batch["summary"],
      max_length=128,
      truncation=True
  )
  return{
      'input_ids' : input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels': target_encodings['input_ids']
  }

In [14]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features , batched = True)

Map: 100%|██████████| 819/819 [00:00<00:00, 7496.37 examples/s]


In [15]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [16]:
#Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collector = DataCollatorForSeq2Seq(tokenizer , model = model_pegasus)


In [17]:
from transformers import TrainingArguments

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",   # v5 syntax
    eval_steps=500,
    save_steps=1_000_000,
    gradient_accumulation_steps=16
)

In [ ]:
from transformers import TrainingArguments , Trainer

# trainer = Trainer(
#     model=model_pegasus,
#     args=trainer_args,
#     data_collator=seq2seq_data_collator,
#     train_dataset=dataset_samsum_pt["train"],
#     eval_dataset=dataset_samsum_pt["validation"]
# )

from transformers import DataCollatorForSeq2Seq

# Create a data collator for seq2seq tasks
seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_pegasus
)

# Now you can safely pass it into Trainer
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"]
)



In [22]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [67]:
#Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches we can process simultaneously.
    Yield successive bacth-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="dialogue",
                                column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                           padding="max_length", return_tensors="pt")

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )
        '''parameter for length penalty ensures that the model does not generate sequences'''

        # Decode generated summaries
        decoded_summaries = tokenizer.batch_decode(summaries, skip_special_tokens=True)
        # decoded_summaries= [d.replace(""," ") for d in decoded_summaries]

        # Add batch to metric
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Compute and return ROUGE scores
    score = metric.compute()
    return score

In [68]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

In [72]:
score = calculate_metric_on_test_ds(
    dataset_samsum_pt["test"], rouge_metric, trainer.model, tokenizer, batch_size=2 , column_text="dialogue" , column_summary="summary"
)
rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=["pegasus"])


100%|██████████| 410/410 [23:30<00:00,  3.44s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.447005,0.215296,0.353888,0.353957


In [73]:
# Save model
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [74]:
#Save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [75]:
# Load

tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [83]:
# Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

sample_text = dataset_samsum_pt["test"][0]["dialogue"]
reference = dataset_samsum_pt["test"][0]["summary"]

# Encode the input dialogue and move to GPU if available
inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=1024
).to(trainer.model.device)   # match model device

# Generate summary
summary_ids = trainer.model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    **gen_kwargs
)

# Decode the generated summary
model_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(model_summary)




Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda has Betty's number. She can't find Betty's number. He called her last time they were at the park together. Hannah doesn't know him well.
